In [1]:
pip install pandas numpy scikit-learn matplotlib seaborn nltk spacy tensorflow transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of mkl-fft to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of mkl-fft to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 45.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 95.0 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: blis
    Found existing installation: blis 1.3.0
    Uninstalling blis-1.3.0:
      Successfully uninstalled blis-1.3.0
  Attempting uninstall: thinc
    Found existing installation: t

In [2]:
!pip install catboost

In [3]:
import pandas as pd
import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from scipy.sparse import hstack

# ============================
# 1. Load Dataset
# ============================
df = pd.read_csv("/kaggle/input/update/IMDB_Cleaned.csv")  # change path here
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

from sklearn.model_selection import train_test_split
X = df['review']
y = df['sentiment']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# subset sizes (can change)
train_size, valid_size, test_size = 7000, 2000, 1000
X_train = X_train.iloc[:train_size].reset_index(drop=True)
y_train = y_train.iloc[:train_size].reset_index(drop=True)
X_valid = X_valid.iloc[:valid_size].reset_index(drop=True)
y_valid = y_valid.iloc[:valid_size].reset_index(drop=True)
X_test  = X_test.iloc[:test_size].reset_index(drop=True)
y_test  = y_test.iloc[:test_size].reset_index(drop=True)

print("Data Sizes → Train:", len(X_train), "Valid:", len(X_valid), "Test:", len(X_test))

# ============================
# 2. TF-IDF Features
# ============================
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_valid_tfidf = tfidf.transform(X_valid)
X_test_tfidf  = tfidf.transform(X_test)

# ============================
# 3. RoBERTa Embeddings
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModel.from_pretrained("roberta-base").to(device)

def roberta_encode(texts, tokenizer, model, device, max_len=128):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), 32):  # batch size = 32
            batch = texts[i:i+32]
            tokens = tokenizer(batch, padding=True, truncation=True,
                               max_length=max_len, return_tensors="pt").to(device)
            outputs = model(**tokens)
            mask = tokens['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
            summed = torch.sum(outputs.last_hidden_state * mask, 1)
            counts = torch.clamp(mask.sum(1), min=1e-9)
            mean_pooled = summed / counts
            embeddings.append(mean_pooled.cpu().numpy())
    return np.vstack(embeddings)

X_train_roberta = roberta_encode(X_train.tolist(), tokenizer, model, device)
X_valid_roberta = roberta_encode(X_valid.tolist(), tokenizer, model, device)
X_test_roberta  = roberta_encode(X_test.tolist(), tokenizer, model, device)

print("RoBERTa embeddings shape:", X_train_roberta.shape)

# ============================
# 4. Hybrid Features (Concat)
# ============================
X_train_hybrid = hstack([X_train_tfidf, X_train_roberta])
X_valid_hybrid = hstack([X_valid_tfidf, X_valid_roberta])
X_test_hybrid  = hstack([X_test_tfidf,  X_test_roberta])

print("Hybrid feature shape:", X_train_hybrid.shape)

# ============================
# 5. Classification Models
# ============================
models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1),
    "RandomForest": RandomForestClassifier(n_estimators=200, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8, 
                             use_label_encoder=False, eval_metric="logloss"),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# ============================
# 6. Training & Evaluation
# ============================
results = []

for name, clf in models.items():
    print(f"\n🔹 Training {name} ...")
    clf.fit(X_train_hybrid, y_train)

    # Validation
    val_pred = clf.predict(X_valid_hybrid)
    val_acc = accuracy_score(y_valid, val_pred)
    val_f1  = f1_score(y_valid, val_pred, average="weighted")
    val_prec = precision_score(y_valid, val_pred, average="weighted")
    val_rec  = recall_score(y_valid, val_pred, average="weighted")

    # Test
    test_pred = clf.predict(X_test_hybrid)
    test_acc = accuracy_score(y_test, test_pred)
    test_f1  = f1_score(y_test, test_pred, average="weighted")
    test_prec = precision_score(y_test, test_pred, average="weighted")
    test_rec  = recall_score(y_test, test_pred, average="weighted")

    results.append((name,
                    val_acc, val_prec, val_rec, val_f1,
                    test_acc, test_prec, test_rec, test_f1))

    print(f"   Validation → Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}")
    print(f"   Test       → Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}, F1: {test_f1:.4f}")

# ============================
# 7. Results DataFrame
# ============================
results_df = pd.DataFrame(
    results,
    columns=['Model',
             'Val_Acc','Val_Prec','Val_Rec','Val_F1',
             'Test_Acc','Test_Prec','Test_Rec','Test_F1']
)

print("\n🏆 Final Results:\n", results_df.sort_values(by='Test_F1', ascending=False))


Data Sizes → Train: 7000 Valid: 2000 Test: 1000


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

2025-08-27 15:31:43.516716: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756308703.855777      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756308703.951826      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RoBERTa embeddings shape: (7000, 768)
Hybrid feature shape: (7000, 5768)

🔹 Training LogReg ...
   Validation → Acc: 0.8845, Prec: 0.8845, Rec: 0.8845, F1: 0.8845
   Test       → Acc: 0.8680, Prec: 0.8682, Rec: 0.8680, F1: 0.8680

🔹 Training RandomForest ...
   Validation → Acc: 0.8440, Prec: 0.8442, Rec: 0.8440, F1: 0.8440
   Test       → Acc: 0.8370, Prec: 0.8382, Rec: 0.8370, F1: 0.8368

🔹 Training GradientBoosting ...
   Validation → Acc: 0.8550, Prec: 0.8551, Rec: 0.8550, F1: 0.8550
   Test       → Acc: 0.8420, Prec: 0.8422, Rec: 0.8420, F1: 0.8419

🔹 Training KNN ...
   Validation → Acc: 0.7560, Prec: 0.7593, Rec: 0.7560, F1: 0.7554
   Test       → Acc: 0.7520, Prec: 0.7536, Rec: 0.7520, F1: 0.7518

🔹 Training XGBoost ...
   Validation → Acc: 0.8660, Prec: 0.8661, Rec: 0.8660, F1: 0.8660
   Test       → Acc: 0.8650, Prec: 0.8658, Rec: 0.8650, F1: 0.8649

🔹 Training CatBoost ...
   Validation → Acc: 0.8670, Prec: 0.8670, Rec: 0.8670, F1: 0.8670
   Test       → Acc: 0.8640, Prec: 0

In [ ]:
# ============================
# 6. Add RoBERTa Classifier
# ============================
from transformers import RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Prepare datasets for HuggingFace Trainer
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
valid_dataset = Dataset.from_dict({"text": X_valid.tolist(), "label": y_valid.tolist()})
test_dataset  = Dataset.from_dict({"text": X_test.tolist(),  "label": y_test.tolist()})

def tokenize_batch(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_batch, batched=True)
valid_dataset = valid_dataset.map(tokenize_batch, batched=True)
test_dataset  = test_dataset.map(tokenize_batch, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
valid_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load classification model
roberta_clf = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

# ✅ FIXED TrainingArguments (use eval_strategy instead of evaluation_strategy)
training_args = TrainingArguments(
    output_dir="./roberta_results",
    eval_strategy="epoch",       # <-- FIXED
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
        "f1": f1_score(labels, preds, average="weighted")
    }

# Trainer
trainer = Trainer(
    model=roberta_clf,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("\n🔹 Training RoBERTa Classifier ...")
trainer.train()

# Evaluate
roberta_eval = trainer.evaluate(test_dataset)
print("   Test →", roberta_eval)


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_36/888177507.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



🔹 Training RoBERTa Classifier ...


<IPython.core.display.Javascript object>